In [5]:
!nvidia-smi

Fri May  8 07:18:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%%writefile vector_add.cu

#include<math.h>
#include<time.h>
#include<iostream>
#include "cuda_runtime.h"

void cpuSum(int* A, int*B, int* C, int N)
    {
    for (int i=0; i<N; ++i)
        {
        C[i] = A[i] + B[i];
        }
    }

__global__ void kernel(int* A, int* B, int* C, int N)
    {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i<N)
        {
        C[i] = A[i] + B[i];
        }
    }

void gpuSum(int* A, int* B, int* C, int N)
    {
    int threadsPerBlock = min(1024, N);
    int blocksPerGrid = ceil(double(N)/double(threadsPerBlock));
    kernel<<<blocksPerGrid, threadsPerBlock>>>(A, B, C, N);
    }

bool isVectorEqual(int* A, int* B, int N)
    {
    for (int i=0; i<N; ++i)
        {
        if (A[i] != B[i])
            {
            return false;
            }
        }
    return true;
    }

int main()
    {
    int N = 2e7;
    int *A, *B, *C, *D, *a, *b, *c;
    int size = N * sizeof(int);

    A = (int*)malloc(size);
    B = (int*)malloc(size);
    C = (int*)malloc(size);
    D = (int*)malloc(size);

    for (int i=0; i<N; ++i)
        {
        A[i] = rand()%1000;
        B[i] = rand()%1000;
        }

    clock_t start, end;

    start = clock();
    cpuSum(A, B, C, N);
    end = clock();
    float timeTakenCPU = ((float)(end-start))/CLOCKS_PER_SEC;

    cudaMalloc(&a, size);
    cudaMalloc(&b, size);
    cudaMalloc(&c, size);

    cudaMemcpy(a, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(b, B, size, cudaMemcpyHostToDevice);

    start = clock();
    gpuSum(a, b, c, N);
    cudaDeviceSynchronize();
    cudaMemcpy(D, c, size, cudaMemcpyDeviceToHost);
    end = clock();
    float timeTakenGPU = ((float)(end-start))/CLOCKS_PER_SEC;

    cudaFree(a);
    cudaFree(b);
    cudaFree(c);

    bool success = isVectorEqual(C, D, N);

    printf("Vector Addition\n");
    printf("--------------------\n");
    printf("CPU Time: %f \n", timeTakenCPU);
    printf("GPU Time: %f \n", timeTakenGPU);
    printf("Speed Up: %f \n", timeTakenCPU/timeTakenGPU);
    printf("Verification: %s \n", success ? "true":"false");
    }

Writing vector_add.cu


In [7]:
!nvcc FLP5HPC4A.cu -o vector_add && ./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Vector Addition
--------------------
CPU Time: 0.091514 
GPU Time: 0.052691 
Speed Up: 1.736805 
Verification: true 
